In [ ]:
from app.services.features import (
    NUMERIC_FEATURES,
    build_training_rows_from_transactions,
    normalize_transactions_frame,
)

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


CSV_PATH = 'transactions.csv'
MODEL_PATH = 'app/ml/subscription_model.joblib'
TRAINING_CANDIDATES_PATH = 'training_candidates.csv'

In [ ]:
raw = pd.read_csv(CSV_PATH, sep=';')
transactions = normalize_transactions_frame(raw)
transactions.head()

In [ ]:
dataset = build_training_rows_from_transactions(transactions)

dataset[['merchant_name', 'mcc_code', 'n_payments', 'interval_mean',
         'amount_cv', 'regularity_score', 'is_subscription']].head(10)

In [ ]:
dataset['is_subscription'].value_counts(dropna=False)

In [ ]:
def build_pipeline(c: float = 1.0, penalty: str = 'l2', class_weight='balanced') -> Pipeline:
    solver = 'liblinear' if penalty in {'l1', 'l2'} else 'lbfgs'
    preprocess = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), NUMERIC_FEATURES),
            ('merchant', TfidfVectorizer(analyzer='char_wb',
             ngram_range=(3, 5), min_df=1), 'merchant_name'),
        ]
    )
    return Pipeline(
        steps=[
            ('preprocess', preprocess),
            ('classifier', LogisticRegression(C=c, penalty=penalty,
             solver=solver, class_weight=class_weight, max_iter=1500)),
        ]
    )